# Piloto de carga controlada SPIDER por pares

## Objetivo y seguridad
No se sobrescriben objetos; no se eliminan objetos; cada carga usa precondicion de generacion; el lote se selecciona por pares imagen-mascara; metadata queda fuera; AXIAL queda fuera; por defecto no se escribe nada.

## Configuracion segura

## Montaje de Drive

## Validacion del manifest

## Construccion de pares SPIDER

## Revalidacion local

## Autenticacion

## Inspeccion del bucket

## Receipt compartido y reanudacion

## Estado parcial de pares

## Seleccion del piloto

## Token de confirmacion

## Carga controlada

## Verificacion posterior

## Resumen final

## Proximas etapas
SPIDER completo y AXIAL requieren tickets posteriores con lotes separados.

In [ ]:
from __future__ import annotations
import csv, hashlib, json, os, tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import pandas as pd

UPLOAD_ENABLED = False
CONFIRM_UPLOAD = ""
TARGET_COMPONENTS = ("spider_image", "spider_mask")
TARGET_PAIR_COUNT = 3
MAX_FILES_THIS_RUN = 6
FAIL_FAST = True
PROJECT_ID = "pfi-asplanatti-fabrello-v1"
BUCKET_NAME = "pfi-rm-lumbar-artifacts-2026-ef"
BUCKET_URI_PREFIX = f"gs://{BUCKET_NAME}/"
GCS_PREFIX = "datasets/"
SPIDER_OBJECT_PREFIX = "datasets/SPIDER/"
ALLOWED_ZERO_BYTE_PLACEHOLDERS = {GCS_PREFIX}
EXPECTED_BUCKET_LOCATION = "US-CENTRAL1"
EXPECTED_BUCKET_STORAGE_CLASS = "STANDARD"
EXPECTED_MANIFEST_SHA256 = "fa54c89a278d850021c0f91c0a27b3b5211c86301c9e4f125d75d517f39b793b"
DRIVE_ROOT = Path("/content/drive")
MY_DRIVE = DRIVE_ROOT / "MyDrive"
PFI_ROOT = MY_DRIVE / "PFI_MVP"
PLAN_DIR = PFI_ROOT / "results" / "GCS_dataset_migration_plan"
UPLOAD_STATE_DIR = PFI_ROOT / "results" / "GCS_dataset_upload"
MANIFEST_CSV = PLAN_DIR / "gcs_upload_manifest.csv"
RECEIPT_CSV = UPLOAD_STATE_DIR / "gcs_upload_receipt.csv"
STATE_JSON = UPLOAD_STATE_DIR / "gcs_upload_state.json"
FAILURES_CSV = UPLOAD_STATE_DIR / "gcs_upload_failures.csv"
EXPECTED_MANIFEST_ROWS = 2058
EXPECTED_TOTAL_BYTES = 3922288649
EXPECTED_SPIDER_IMAGE_ROWS = 447
EXPECTED_SPIDER_MASK_ROWS = 447
EXPECTED_SPIDER_PAIRS = 447
EXPECTED_MANIFEST_COLUMNS = ["component","source_path","source_root","relative_path","destination_uri","suffix","size_bytes","modified_at","sha256","referenced_rows","exists","is_symlink","status"]
RECEIPT_COLUMNS = ["manifest_sha256","component","source_path","relative_path","destination_uri","object_name","size_bytes","crc32c","md5_hash","generation","updated","uploaded_at_utc","upload_status"]
FAILURE_COLUMNS = ["manifest_sha256","component","relative_path","destination_uri","error_type","error_message","failed_at_utc"]

def ensure_drive_available():
    if not MY_DRIVE.exists():
        try:
            from google.colab import drive
        except ImportError as exc:
            raise RuntimeError("Colab o /content/drive/MyDrive requerido") from exc
        drive.mount(str(DRIVE_ROOT))
    if not PFI_ROOT.exists():
        raise FileNotFoundError(f"No existe PFI_ROOT: {PFI_ROOT}")

def sha256_stream(path: Path) -> str:
    with path.open("rb") as fh:
        return hashlib.file_digest(fh, "sha256").hexdigest()

def parse_bool_series(series: pd.Series, name: str) -> pd.Series:
    if series.dtype == bool:
        return series
    normalized = series.astype(str).str.strip().str.lower()
    bad = sorted(set(normalized) - {"true", "false"})
    if bad:
        raise ValueError(f"Columna {name} invalida: {bad[:10]}")
    return normalized == "true"

def path_is_inside(path_text: str, root: Path) -> bool:
    path = Path(path_text)
    return path.is_absolute() and (path == root or root in path.parents)

def destination_object_name(uri: str) -> str:
    if not isinstance(uri, str) or not uri.startswith(BUCKET_URI_PREFIX):
        raise ValueError(f"Destino fuera de bucket: {uri}")
    object_name = uri[len(BUCKET_URI_PREFIX):]
    if not object_name.startswith(GCS_PREFIX):
        raise ValueError(f"Destino fuera de prefijo {GCS_PREFIX}: {uri}")
    return object_name

def normalized_pair_key(path_like: str) -> str:
    stem = Path(path_like).stem.lower()
    for suffix in ("_image", "_mask", "_label"):
        if stem.endswith(suffix):
            stem = stem[:-len(suffix)]
    return stem

def validate_manifest() -> tuple[pd.DataFrame, str]:
    if not MANIFEST_CSV.exists() or not MANIFEST_CSV.is_file():
        raise FileNotFoundError(MANIFEST_CSV)
    manifest_sha256 = sha256_stream(MANIFEST_CSV)
    if manifest_sha256 != EXPECTED_MANIFEST_SHA256:
        raise ValueError(f"Manifest SHA inesperado: {manifest_sha256}")
    m = pd.read_csv(MANIFEST_CSV, keep_default_na=False)
    if list(m.columns) != EXPECTED_MANIFEST_COLUMNS or len(m) != EXPECTED_MANIFEST_ROWS:
        raise ValueError("Manifest con columnas o filas inesperadas")
    m["size_bytes"] = pd.to_numeric(m["size_bytes"], errors="raise").astype("int64")
    m["exists"] = parse_bool_series(m["exists"], "exists")
    m["is_symlink"] = parse_bool_series(m["is_symlink"], "is_symlink")
    checks = [int(m["size_bytes"].sum()) == EXPECTED_TOTAL_BYTES, bool((m["status"].astype(str) == "OK").all()), bool(m["exists"].all()), not bool(m["is_symlink"].any()), not bool((m["size_bytes"] <= 0).any()), not m["destination_uri"].duplicated().any(), not m.duplicated(["component","source_path"]).any(), not bool(m["relative_path"].astype(str).str.strip().eq("").any()), not bool(m["relative_path"].astype(str).str.split("/").map(lambda parts: ".." in parts).any()), not bool(m["suffix"].astype(str).str.lower().eq(".zip").any()), bool(m["source_path"].map(lambda v: path_is_inside(str(v), PFI_ROOT)).all()), bool(m["destination_uri"].astype(str).str.startswith(BUCKET_URI_PREFIX).all())]
    if not all(checks):
        raise ValueError("Manifest invalido para piloto SPIDER")
    return m, manifest_sha256

def build_spider_pairs(m: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    spider = m[m["component"].isin(TARGET_COMPONENTS)].copy()
    counts = spider["component"].value_counts().to_dict()
    if int(counts.get("spider_image",0)) != EXPECTED_SPIDER_IMAGE_ROWS or int(counts.get("spider_mask",0)) != EXPECTED_SPIDER_MASK_ROWS:
        raise ValueError("Conteos SPIDER inesperados")
    spider["manifest_ordinal"] = spider.index.astype(int)
    spider["pair_key"] = spider["relative_path"].map(normalized_pair_key)
    if spider["destination_uri"].duplicated().any():
        raise ValueError("Destinos SPIDER duplicados")
    pc = spider.groupby(["pair_key","component"]).size().unstack(fill_value=0)
    bad = pc[(pc.get("spider_image",0) != 1) | (pc.get("spider_mask",0) != 1)]
    if len(pc) != EXPECTED_SPIDER_PAIRS or not bad.empty:
        raise ValueError("Pares SPIDER incompletos o duplicados")
    order = spider.groupby("pair_key")["manifest_ordinal"].min().sort_values().reset_index()
    order["pair_order"] = range(len(order))
    spider = spider.merge(order, on="pair_key", how="left").sort_values(["pair_order","manifest_ordinal"]).reset_index(drop=True)
    return spider, order

def revalidate_sources_now(m: pd.DataFrame):
    total = len(m)
    for index, row in enumerate(m.itertuples(index=False), start=1):
        p = Path(row.source_path)
        if not p.exists() or p.is_symlink() or not p.is_file() or p.stat().st_size != int(row.size_bytes):
            raise ValueError(f"Fuente invalida: component={row.component} relative_path={row.relative_path}")
        if index % 250 == 0 or index == total:
            print(f"Fuentes revalidadas: {index}/{total}")

def authenticate_gcp():
    try:
        from google.colab import auth
    except ImportError as exc:
        raise RuntimeError("Autenticacion interactiva de Colab requerida") from exc
    try:
        import google.auth
        from google.cloud import storage
    except ImportError as exc:
        raise RuntimeError("Falta google-cloud-storage/google-auth; instalar manualmente") from exc
    auth.authenticate_user()
    credentials, active_project = google.auth.default()
    print(json.dumps({"client_project": PROJECT_ID, "auth_project_detected": active_project}, indent=2))
    return storage.Client(project=PROJECT_ID, credentials=credentials)

def planned_object_names(m: pd.DataFrame) -> set[str]:
    return set(m["destination_uri"].map(destination_object_name))

def inspect_bucket(client):
    bucket = client.bucket(BUCKET_NAME)
    bucket.reload(client=client)
    if bucket.name != BUCKET_NAME or str(bucket.location or "").upper() != EXPECTED_BUCKET_LOCATION or str(bucket.storage_class or "").upper() != EXPECTED_BUCKET_STORAGE_CLASS:
        raise ValueError("Metadata de bucket inesperada")
    planned = planned_object_names(manifest_df)
    existing, placeholders, external = {}, [], []
    for blob in client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX):
        item = {"name": blob.name, "size": int(blob.size or 0), "crc32c": blob.crc32c, "md5_hash": blob.md5_hash, "generation": str(blob.generation) if blob.generation is not None else None, "updated": blob.updated.isoformat() if blob.updated is not None else None}
        existing[blob.name] = item
        if blob.name in ALLOWED_ZERO_BYTE_PLACEHOLDERS and item["size"] == 0:
            placeholders.append(item)
        elif blob.name not in planned:
            external.append(item)
    if external:
        raise RuntimeError(f"Objetos no planificados bajo {GCS_PREFIX}: {external[:10]}")
    return bucket, existing, placeholders, external

def atomic_write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="") as fh:
            fh.write(text); fh.flush(); os.fsync(fh.fileno())
        os.replace(tmp, path)
    finally:
        t = Path(tmp)
        if t.exists():
            t.unlink()

def save_receipt(receipt: pd.DataFrame):
    atomic_write_text(RECEIPT_CSV, receipt[RECEIPT_COLUMNS].to_csv(index=False, quoting=csv.QUOTE_MINIMAL))

def save_failures(failures: pd.DataFrame):
    atomic_write_text(FAILURES_CSV, failures[FAILURE_COLUMNS].to_csv(index=False, quoting=csv.QUOTE_MINIMAL))

def save_state(payload: dict[str, Any]):
    atomic_write_text(STATE_JSON, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")

def load_receipt() -> pd.DataFrame:
    if not RECEIPT_CSV.exists():
        raise FileNotFoundError(f"Falta receipt compartido esperado: {RECEIPT_CSV}")
    r = pd.read_csv(RECEIPT_CSV, keep_default_na=False)
    if list(r.columns) != RECEIPT_COLUMNS or r["destination_uri"].duplicated().any():
        raise ValueError("Receipt incompatible con notebook 40")
    if not r.empty and not bool((r["manifest_sha256"] == MANIFEST_SHA256).all()):
        raise ValueError("Receipt ligado a otro manifest")
    return r

def load_failures() -> pd.DataFrame:
    if not FAILURES_CSV.exists():
        return pd.DataFrame(columns=FAILURE_COLUMNS)
    f = pd.read_csv(FAILURES_CSV, keep_default_na=False)
    if list(f.columns) != FAILURE_COLUMNS:
        raise ValueError("Failures incompatible")
    return f

def receipt_lookup(r: pd.DataFrame) -> dict[str, dict[str, Any]]:
    return {str(row.destination_uri): row._asdict() for row in r.itertuples(index=False)}

def remote_matches_receipt(row: pd.Series, remote: dict[str, Any] | None, rec: dict[str, Any] | None) -> bool:
    return bool(remote is not None and rec is not None and str(rec["manifest_sha256"]) == MANIFEST_SHA256 and str(rec["destination_uri"]) == str(row["destination_uri"]) and int(rec["size_bytes"]) == int(remote["size"]) and str(rec["generation"]) == str(remote["generation"]) and str(rec["crc32c"]) == str(remote["crc32c"]))

def classify_plan_status(m: pd.DataFrame, existing: dict[str, dict[str, Any]], receipt: pd.DataFrame) -> pd.DataFrame:
    receipts = receipt_lookup(receipt); rows = []
    for ordinal, row in enumerate(m.itertuples(index=False)):
        rec = row._asdict(); object_name = destination_object_name(str(rec["destination_uri"])); remote = existing.get(object_name); receipt_row = receipts.get(str(rec["destination_uri"]))
        status = "TO_UPLOAD" if remote is None else ("ALREADY_UPLOADED_VERIFIED" if remote_matches_receipt(pd.Series(rec), remote, receipt_row) else "EXISTING_UNVERIFIED")
        rows.append({"manifest_ordinal": ordinal, "component": rec["component"], "source_path": rec["source_path"], "relative_path": rec["relative_path"], "destination_uri": rec["destination_uri"], "object_name": object_name, "size_bytes": int(rec["size_bytes"]), "plan_status": status})
    return pd.DataFrame(rows)

def classify_spider_pairs(spider: pd.DataFrame, classified: pd.DataFrame):
    ss = spider.merge(classified[["destination_uri","plan_status"]], on="destination_uri", how="left", validate="one_to_one")
    rows = []
    for pair_key, g in ss.groupby("pair_key", sort=False):
        statuses = set(g["plan_status"])
        if "EXISTING_UNVERIFIED" in statuses or g["plan_status"].isna().any() or len(g) != 2:
            pair_status = "PAIR_BLOCKED"
        elif statuses == {"TO_UPLOAD"}:
            pair_status = "PAIR_PENDING"
        elif statuses == {"ALREADY_UPLOADED_VERIFIED"}:
            pair_status = "PAIR_COMPLETE_VERIFIED"
        elif statuses == {"TO_UPLOAD", "ALREADY_UPLOADED_VERIFIED"}:
            pair_status = "PAIR_PARTIAL_VERIFIED"
        else:
            pair_status = "PAIR_BLOCKED"
        rows.append({"pair_key": pair_key, "pair_order": int(g["pair_order"].min()), "pair_status": pair_status, "pending_files": int((g["plan_status"] == "TO_UPLOAD").sum()), "verified_files": int((g["plan_status"] == "ALREADY_UPLOADED_VERIFIED").sum())})
    ps = pd.DataFrame(rows).sort_values("pair_order").reset_index(drop=True)
    blocked = ps[ps["pair_status"] == "PAIR_BLOCKED"]
    if not blocked.empty:
        raise RuntimeError(f"Pares SPIDER bloqueados: {blocked.head(10).to_dict('records')}")
    return ss, ps

ensure_drive_available()
manifest_df, MANIFEST_SHA256 = validate_manifest(); MANIFEST_READY = True
spider_manifest_df, spider_pair_order_df = build_spider_pairs(manifest_df); PAIR_MANIFEST_READY = True
revalidate_sources_now(manifest_df); SOURCE_FILES_READY = True
storage_client = authenticate_gcp()
bucket, existing_objects, allowed_placeholders, unplanned_existing_objects = inspect_bucket(storage_client); BUCKET_READY = True
receipt_df = load_receipt(); failures_df = load_failures(); failures_this_run: list[dict[str, Any]] = []
classified_manifest_df = classify_plan_status(manifest_df, existing_objects, receipt_df)
existing_unverified_df = classified_manifest_df[classified_manifest_df["plan_status"] == "EXISTING_UNVERIFIED"]
if not existing_unverified_df.empty:
    raise RuntimeError(f"Objetos planificados existentes sin receipt verificable: {existing_unverified_df.head(10).to_dict('records')}")
metadata_verified_df = classified_manifest_df[classified_manifest_df["component"].isin(["metadata_e5","metadata_e9"]) & (classified_manifest_df["plan_status"] == "ALREADY_UPLOADED_VERIFIED")]
METADATA_RECEIPTS_VERIFIED = bool(set(metadata_verified_df["component"]) == {"metadata_e5", "metadata_e9"} and len(metadata_verified_df) == 2)
if not METADATA_RECEIPTS_VERIFIED:
    raise RuntimeError("Metadata E5/E9 no verificada exactamente por receipt")
spider_status_df, pair_status_df = classify_spider_pairs(spider_manifest_df, classified_manifest_df)
eligible_pairs_df = pair_status_df[pair_status_df["pair_status"] != "PAIR_COMPLETE_VERIFIED"].head(TARGET_PAIR_COUNT).copy()
selected_pair_keys = list(eligible_pairs_df["pair_key"])
selected_df = spider_status_df[spider_status_df["pair_key"].isin(selected_pair_keys) & (spider_status_df["plan_status"] == "TO_UPLOAD")].sort_values(["pair_order","manifest_ordinal"]).copy()
selected_pairs = int(len(eligible_pairs_df)); selected_files = int(len(selected_df)); selected_bytes = int(selected_df["size_bytes"].sum()) if selected_files else 0
if selected_pairs <= 0 or selected_pairs > TARGET_PAIR_COUNT or selected_files <= 0 or selected_files > MAX_FILES_THIS_RUN:
    raise RuntimeError("Seleccion piloto invalida")
initial_spider_receipts = int((spider_status_df["plan_status"] == "ALREADY_UPLOADED_VERIFIED").sum())
if initial_spider_receipts == 0:
    ppc = selected_df.groupby(["pair_key","component"]).size().unstack(fill_value=0)
    if selected_pairs != TARGET_PAIR_COUNT or selected_files != MAX_FILES_THIS_RUN or not bool((ppc["spider_image"] == 1).all() and (ppc["spider_mask"] == 1).all()):
        raise RuntimeError("Piloto inicial SPIDER inesperado")
print(selected_df[["pair_key","component","relative_path","destination_uri","size_bytes","plan_status"]].to_string(index=False))
CONFIRMATION_TOKEN = f"UPLOAD_SPIDER_{selected_pairs}_PAIRS_{selected_files}_FILES_{selected_bytes}_BYTES_TO_{BUCKET_NAME}_{MANIFEST_SHA256[:12]}"
UPLOAD_ARMED = bool(UPLOAD_ENABLED is True and CONFIRM_UPLOAD == CONFIRMATION_TOKEN and MANIFEST_READY and PAIR_MANIFEST_READY and SOURCE_FILES_READY and BUCKET_READY and METADATA_RECEIPTS_VERIFIED and not unplanned_existing_objects and existing_unverified_df.empty and selected_pairs > 0 and selected_pairs <= TARGET_PAIR_COUNT and selected_files > 0 and selected_files <= MAX_FILES_THIS_RUN)
print(json.dumps({"required_confirmation_token": CONFIRMATION_TOKEN, "UPLOAD_ENABLED": UPLOAD_ENABLED, "UPLOAD_ARMED": UPLOAD_ARMED}, indent=2))

def append_failure(failures: pd.DataFrame, row: pd.Series, error_type: str, error_message: str) -> pd.DataFrame:
    item = {"manifest_sha256": MANIFEST_SHA256, "component": row["component"], "relative_path": row["relative_path"], "destination_uri": row["destination_uri"], "error_type": error_type, "error_message": str(error_message)[:500], "failed_at_utc": datetime.now(timezone.utc).isoformat()}
    failures_this_run.append(item)
    failures = pd.concat([failures, pd.DataFrame([item])], ignore_index=True)
    save_failures(failures)
    return failures

def upload_one_spider_file(bucket: Any, row: pd.Series) -> dict[str, Any]:
    if row["component"] not in TARGET_COMPONENTS:
        raise ValueError(f"Componente no permitido: {row['component']}")
    source_path = Path(row["source_path"])
    if not source_path.exists() or source_path.is_symlink() or not source_path.is_file() or source_path.stat().st_size != int(row["size_bytes"]):
        raise ValueError(f"Fuente invalida: pair_key={row['pair_key']} relative_path={row['relative_path']}")
    object_name = destination_object_name(str(row["destination_uri"]))
    if not object_name.startswith(SPIDER_OBJECT_PREFIX):
        raise ValueError(f"Objeto SPIDER fuera de prefijo permitido: {object_name}")
    blob = bucket.blob(object_name)
    blob.metadata = {"pfi_manifest_sha256": MANIFEST_SHA256, "pfi_component": str(row["component"]), "pfi_pair_key": str(row["pair_key"]), "pfi_relative_path": str(row["relative_path"]), "pfi_source_size": str(int(row["size_bytes"])), "pfi_upload_notebook": "41"}
    blob.upload_from_filename(
        str(source_path),
        if_generation_match=0,
        checksum="auto",
        timeout=900,
    )
    blob.reload()
    if int(blob.size or 0) != int(row["size_bytes"]) or blob.generation is None or blob.crc32c is None:
        raise ValueError(f"Verificacion remota fallo: {object_name}")
    return {"manifest_sha256": MANIFEST_SHA256, "component": row["component"], "source_path": row["source_path"], "relative_path": row["relative_path"], "destination_uri": row["destination_uri"], "object_name": object_name, "size_bytes": int(row["size_bytes"]), "crc32c": blob.crc32c, "md5_hash": blob.md5_hash, "generation": str(blob.generation), "updated": blob.updated.isoformat() if blob.updated is not None else "", "uploaded_at_utc": datetime.now(timezone.utc).isoformat(), "upload_status": "UPLOADED_VERIFIED"}

uploaded_receipts: list[dict[str, Any]] = []
if UPLOAD_ARMED:
    try:
        from google.api_core.exceptions import PreconditionFailed
        from google.cloud.storage.exceptions import DataCorruption
    except ImportError as exc:
        raise RuntimeError("Faltan excepciones de google-cloud-storage/google-api-core en el entorno") from exc
    for row in selected_df.to_dict("records"):
        row_series = pd.Series(row)
        try:
            item = upload_one_spider_file(bucket, row_series)
            uploaded_receipts.append(item)
            receipt_df = pd.concat([receipt_df, pd.DataFrame([item])], ignore_index=True)
            save_receipt(receipt_df)
            save_state({"manifest_sha256": MANIFEST_SHA256, "notebook": "41", "last_pair_key": row["pair_key"], "last_uploaded_destination_uri": row["destination_uri"], "updated_at_utc": datetime.now(timezone.utc).isoformat()})
            print(f"Cargado y verificado: pair_key={row['pair_key']} component={row['component']}")
        except (PreconditionFailed, DataCorruption) as exc:
            failures_df = append_failure(failures_df, row_series, type(exc).__name__, str(exc))
            if FAIL_FAST: raise
        except Exception as exc:
            failures_df = append_failure(failures_df, row_series, type(exc).__name__, str(exc))
            if FAIL_FAST: raise
else:
    print("UPLOAD_ARMED=False; se omite la carga serial SPIDER.")

def list_current_objects(client):
    return {blob.name: {"size": int(blob.size or 0), "crc32c": blob.crc32c, "generation": str(blob.generation) if blob.generation is not None else None} for blob in client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX)}

def verify_uploaded_batch(client, uploaded_items: list[dict[str, Any]]) -> int:
    if not uploaded_items: return 0
    current = list_current_objects(client); planned = planned_object_names(manifest_df)
    extras = [name for name, remote in current.items() if not (name in ALLOWED_ZERO_BYTE_PLACEHOLDERS and int(remote["size"]) == 0) and name not in planned]
    if extras: raise RuntimeError(f"Objetos ajenos bajo {GCS_PREFIX}: {extras[:10]}")
    verified = 0
    for item in uploaded_items:
        remote = current.get(item["object_name"])
        if remote is None or int(remote["size"]) != int(item["size_bytes"]) or str(remote["generation"]) != str(item["generation"]) or str(remote["crc32c"]) != str(item["crc32c"]):
            raise RuntimeError(f"Verificacion final no coincide: {item['object_name']}")
        verified += 1
    return verified

verified_this_run = verify_uploaded_batch(storage_client, uploaded_receipts)
if uploaded_receipts:
    refreshed_classified_df = classify_plan_status(manifest_df, list_current_objects(storage_client), load_receipt())
    refreshed_spider_status_df, refreshed_pair_status_df = classify_spider_pairs(spider_manifest_df, refreshed_classified_df)
    selected_pair_status = refreshed_pair_status_df[refreshed_pair_status_df["pair_key"].isin(selected_pair_keys)]
    if not bool((selected_pair_status["pair_status"] == "PAIR_COMPLETE_VERIFIED").all()):
        raise RuntimeError("Pares seleccionados no quedaron completos")
else:
    refreshed_pair_status_df = pair_status_df

final_summary = {"manifest_sha256": MANIFEST_SHA256, "target_components": list(TARGET_COMPONENTS), "target_pair_count": TARGET_PAIR_COUNT, "max_files_this_run": MAX_FILES_THIS_RUN, "selected_pairs": selected_pairs, "selected_files": selected_files, "selected_bytes": selected_bytes, "uploaded_this_run": len(uploaded_receipts), "verified_this_run": verified_this_run, "already_uploaded_verified_files": int((classified_manifest_df["plan_status"] == "ALREADY_UPLOADED_VERIFIED").sum()), "fully_verified_spider_pairs": int((refreshed_pair_status_df["pair_status"] == "PAIR_COMPLETE_VERIFIED").sum()), "partial_verified_spider_pairs": int((refreshed_pair_status_df["pair_status"] == "PAIR_PARTIAL_VERIFIED").sum()), "remaining_spider_pairs": int((refreshed_pair_status_df["pair_status"] != "PAIR_COMPLETE_VERIFIED").sum()), "metadata_receipts_verified": bool(METADATA_RECEIPTS_VERIFIED), "failures_this_run": len(failures_this_run), "cumulative_failure_rows": int(len(failures_df)), "placeholder_count": len(allowed_placeholders), "UPLOAD_BATCH_SUCCESS": bool(UPLOAD_ARMED and len(uploaded_receipts) == selected_files and verified_this_run == selected_files and len(failures_this_run) == 0)}
print(json.dumps(final_summary, indent=2))